# Qwen3 Position-Only Factorial Diagnostic
## v0.6.1 — prospective follow-up to the failed directional rescue hypothesis

v0.6.0 **did not support the predeclared degradation → rescue mechanism**. The pooled forced-choice-NLL fragmentation effect had the opposite sign, and neither block-affine nor source-aligned positions produced a positive pooled rescue.

That failed directional result remains a failed result.

However, v0.6.0 exposed a more specific mechanistic possibility:

> the expanded positional geometry itself may be changing retrieval behavior, independently of the fragmented token identities.

This notebook tests that explanation prospectively on **32 entirely new semantic seeds**.

## Missing factorial cell

v0.6.0 had:

- canonical IDs + canonical positions;
- fragmented IDs + expanded positions;
- fragmented IDs + source-aligned positions.

It did **not** have:

- canonical IDs + expanded positions.

v0.6.1 adds that missing cell and forms a clean 2×2 design:

| Token IDs | Canonical/source geometry | Expanded geometry |
|---|---|---|
| canonical | `CC` | `CE` |
| fragmented filler | `FC` | `FE` |

Where:

- `CC = canonical_ids_canonical_pos`
- `CE = canonical_ids_expanded_pos`
- `FC = fragmented_ids_source_pos`
- `FE = fragmented_ids_expanded_pos`

For `CE`, the canonical filler tokens are **positionally stretched** across exactly the same global filler span used by `FE`; the query and assistant suffix receive the same absolute positions as in `FE`.

Thus `CC` and `CE` have identical token IDs and differ only in position IDs.  
Likewise `FC` and `FE` have identical token IDs and differ only in position IDs.

## Primary readout

The primary metric is now the **gold-vs-best-distractor candidate logit margin**:

`gold_logit - max(other legal digit logits)`

Higher is better.

This avoids the numerical ceiling seen in forced-choice NLL when the correct probability rounds to 1.0. Forced-choice NLL remains a secondary continuity metric.

## New data only

Seeds `2000..2031` are new. They were not used in v0.5 calibration or v0.6.0 mechanism testing.

Gold digit × target-position rank is exactly balanced: each of the 16 combinations occurs twice.

## Interpretation

The main prospective tests are:

- `position_effect_canonical_ids = CE - CC`
- `position_effect_fragmented_ids = FE - FC`

If both effects reproduce the same direction, the v0.6.0 sign reversal is attributable to **position expansion itself**, not merely to extra fragmented filler tokens.

This notebook is a follow-up diagnostic. It does not retroactively convert the failed v0.6.0 degradation/rescue hypothesis into a success.

> Public reproduction copy: calibration values are frozen explicitly; calibration scans and development/debugging history are not required to reproduce the paper's reported experiment.

## 0. T4 runtime

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 1. Drive, provenance, and frozen settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, shutil

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3_position_factorial_v061')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = PROJECT_DIR / 'v061_factorial_4k8k.jsonl'
AUDIT_PATH = PROJECT_DIR / 'v061_audit.json'
ZIP_PATH = PROJECT_DIR / 'fragburden0.3.0.zip'
REPO_DIR = PROJECT_DIR / 'fragburden_src' / 'fragburden'

MODEL_NAME = 'Qwen/Qwen3-4B'
TRANSFORMERS_VERSION = '5.15.1'
TASK_VERSION = 'symbolic_assoc_r1_v061'
SCORER_VERSION = 'assoc_chunked_margin_v061'
POSITION_INTERVENTION = 'canonical_vs_fragmented_x_source_vs_expanded_v1'
PREFILL_CHUNK = 512
TARGET_LENGTHS = [4096, 8192]
FILLER_LANGUAGES = ['en', 'tr']
SEMANTIC_SEEDS = [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031]
POSITION_RANK_SET = [0, 2, 5, 7]
BOOTSTRAP_REPS = 10000
BOOTSTRAP_SEED = 20260828

# Frozen during outcome-independent calibration and reported in the paper.
N_RECORDS = 32
FRAG_KEEP_BY_LANGUAGE = {'en': 2, 'tr': 5}
previous = {'mechanism_go': False}

print('FROZEN PAPER SETTINGS')
print('N_RECORDS =', N_RECORDS)
print('FRAG_KEEP_BY_LANGUAGE =', FRAG_KEEP_BY_LANGUAGE)
print('SEMANTIC_SEEDS =', SEMANTIC_SEEDS)


## 2. Install `fragburden` and validated Transformers

In [ ]:
import subprocess, sys

if not ZIP_PATH.exists():
    repo_zip = Path('/content/token-distance-positional-geometry/code/fragburden0.3.0.zip')
    if repo_zip.exists():
        shutil.copy2(repo_zip, ZIP_PATH)
        print('Copied fragburden from cloned repository.')
    else:
        from google.colab import files
        uploaded = files.upload()
        zip_names = [n for n in uploaded if n.endswith('.zip')]
        if not zip_names:
            raise RuntimeError('Upload code/fragburden0.3.0.zip from the repository.')
        shutil.copy2(Path(zip_names[0]), ZIP_PATH)

extract_root = PROJECT_DIR / 'fragburden_src'
if extract_root.exists(): shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(ZIP_PATH), str(extract_root))
assert REPO_DIR.exists() and (REPO_DIR / 'pyproject.toml').exists()
subprocess.check_call([sys.executable,'-m','pip','install','-e',f'{REPO_DIR}[gpu,analysis]'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-U',f'transformers=={TRANSFORMERS_VERSION}','accelerate','bitsandbytes'])
print('Installation complete.')


## 3. Imports and model

In [ ]:
import hashlib, random, math, inspect, sys, gc, time
from dataclasses import dataclass

import numpy as np
import pandas as pd

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch
import torch.nn.functional as F
import transformers

import fragburden
from fragburden.dial import FragmentationDial, verify_monotone
from fragburden.testing import sample_texts

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DynamicCache,
)

assert transformers.__version__ == TRANSFORMERS_VERSION
assert torch.cuda.is_available()

print('fragburden:', fragburden.__file__)
print('transformers:', transformers.__version__)
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

load_kwargs = dict(
    quantization_config=bnb,
    device_map='auto',
    attn_implementation='sdpa',
)

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16,
        **load_kwargs,
    )
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        **load_kwargs,
    )

model.eval()

if hasattr(model.config, 'use_cache'):
    model.config.use_cache = False

backbone = model.model
device = next(model.parameters()).device

if 'position_ids' not in inspect.signature(backbone.forward).parameters:
    raise RuntimeError('Backbone does not expose position_ids.')

print('Loaded:', MODEL_NAME)

## 4. Fragmentation dial gate

In [ ]:
dial = FragmentationDial.from_transformers(
    tokenizer,
    name=MODEL_NAME,
)

cap = dial.capability()

dial_gate = verify_monotone(
    dial,
    sample_texts('en') + sample_texts('tr'),
    levels=(100, 50, 25, 10, 5, 2, 1),
)

print(json.dumps(cap, indent=2))
print(json.dumps(dial_gate, indent=2))

if not cap.get('usable_for_arm_a'):
    raise RuntimeError('No usable fragmentation dial.')

if (
    not dial_gate['monotone_nondecreasing']
    or not dial_gate['all_in_vocab']
):
    raise RuntimeError('Fragmentation dial failed.')

d100 = dial.at(100)

DIALS = {
    language: dial.at(
        FRAG_KEEP_BY_LANGUAGE[language]
    )
    for language in FILLER_LANGUAGES
}

print('DIAL GATE PASSED.')

## 5. Official non-thinking chat shell and digit candidates

In [ ]:
DIGITS = list('1234')

def _normalize_input_ids(obj):
    if hasattr(obj, 'keys') and 'input_ids' in obj:
        obj = obj['input_ids']

    if torch.is_tensor(obj):
        obj = obj.detach().cpu().tolist()

    if (
        isinstance(obj, (list, tuple))
        and len(obj) == 1
        and isinstance(obj[0], (list, tuple))
    ):
        obj = obj[0]

    return [int(x) for x in obj]

def apply_chat_ids(user_content):
    out = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_content}],
        tokenize=True,
        return_dict=True,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    return _normalize_input_ids(out)

def render_chat(user_content):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_content}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

DIGIT_IDS = {}

for d in DIGITS:
    ids = tokenizer(
        d,
        add_special_tokens=False,
    )['input_ids']

    if len(ids) != 1:
        raise RuntimeError(f'{d} is not one token.')

    DIGIT_IDS[d] = int(ids[0])

if len(set(DIGIT_IDS.values())) != 4:
    raise RuntimeError('Digit tokens are not distinct.')

SHELL_MARKER = 'UNIQUE_USER_CONTENT_MARKER_V061_A93F27'

rendered_marker = render_chat(SHELL_MARKER)

if rendered_marker.count(SHELL_MARKER) != 1:
    raise RuntimeError('Could not isolate user-content shell.')

marker_idx = rendered_marker.index(SHELL_MARKER)

CHAT_PREFIX_TEXT = rendered_marker[:marker_idx]
CHAT_SUFFIX_TEXT = rendered_marker[
    marker_idx + len(SHELL_MARKER):
]

CHAT_PREFIX_IDS = tokenizer(
    CHAT_PREFIX_TEXT,
    add_special_tokens=False,
)['input_ids']

CHAT_SUFFIX_IDS = tokenizer(
    CHAT_SUFFIX_TEXT,
    add_special_tokens=False,
)['input_ids']

def canonical_user_ids(user_content):
    ids_tok = tokenizer(
        user_content,
        add_special_tokens=False,
    )['input_ids']

    ids_dial = d100.pieces_to_ids(
        d100.encode_pieces(user_content)
    )

    if ids_tok != ids_dial:
        raise RuntimeError(
            'Canonical dial IDs disagree with tokenizer.'
        )

    return ids_dial

def verify_chat_shell(user_content):
    user_ids = canonical_user_ids(user_content)

    reconstructed = (
        CHAT_PREFIX_IDS
        + user_ids
        + CHAT_SUFFIX_IDS
    )

    official = apply_chat_ids(user_content)

    if reconstructed != official:
        raise RuntimeError('Chat-shell reconstruction failed.')

    return {
        'official_ids': official,
        'canonical_user_ids': user_ids,
    }

print('Digit IDs:', DIGIT_IDS)
print('Prefix tokens:', len(CHAT_PREFIX_IDS))
print('Suffix tokens:', len(CHAT_SUFFIX_IDS))
print('Chat shell ready.')

## 6. New 32-seed balanced associative-retrieval generator

In [ ]:
@dataclass
class CharSpan:
    start: int
    end: int
    label: str

@dataclass
class AssocInstance:
    filler_language: str
    semantic_seed: int
    trial_index: int
    target_chat_tokens: int
    user_content: str
    spans: list
    target_key: str
    gold_digit: str
    target_record_position: int
    evidence_lines: list
    filler_lines: list
    query_lines: list

def stable_int(*parts):
    blob = '|'.join(map(str, parts)).encode('utf-8')
    return int.from_bytes(
        hashlib.sha256(blob).digest()[:8],
        'big',
    )

def stable_rng(*parts):
    return random.Random(stable_int(*parts))

def make_key(semantic_seed, slot):
    value = (
        stable_int(
            'key_v061',
            semantic_seed,
            slot,
            N_RECORDS,
        )
        % 90000
        + 10000
    )

    return f'K{value}'

def gold_for_trial(trial_index):
    return DIGITS[trial_index % 4]

def position_rank_for_trial(trial_index):
    gold_index = trial_index % 4
    replicate = trial_index // 4

    return POSITION_RANK_SET[
        (replicate + gold_index)
        % len(POSITION_RANK_SET)
    ]

def target_position_for_trial(trial_index):
    rank = position_rank_for_trial(trial_index)

    return int(
        round(
            rank
            * (N_RECORDS - 1)
            / 7
        )
    )

FILLER_BANK = {
    'en': {
        'subjects': [
            'The archivist',
            'The technician',
            'The editor',
            'The visitor',
            'The clerk',
            'The gardener',
        ],
        'verbs': [
            'reviewed',
            'checked',
            'organized',
            'recorded',
            'sorted',
            'noted',
        ],
        'objects': [
            'a routine report',
            'the daily schedule',
            'a short memo',
            'the supply list',
            'a maintenance note',
            'the meeting agenda',
        ],
        'settings': [
            'near the window',
            'in the quiet office',
            'before lunch',
            'during the afternoon',
            'at the front desk',
            'after the meeting',
        ],
    },
    'tr': {
        'subjects': [
            'Arşiv görevlisi',
            'Teknisyen',
            'Editör',
            'Ziyaretçi',
            'Görevli',
            'Bahçıvan',
        ],
        'verbs': [
            'inceledi',
            'kontrol etti',
            'düzenledi',
            'kaydetti',
            'sıraladı',
            'not etti',
        ],
        'objects': [
            'sıradan bir raporu',
            'günlük programı',
            'kısa bir notu',
            'malzeme listesini',
            'bakım notunu',
            'toplantı gündemini',
        ],
        'settings': [
            'pencerenin yanında',
            'sessiz ofiste',
            'öğle yemeğinden önce',
            'öğleden sonra',
            'ön masada',
            'toplantıdan sonra',
        ],
    },
}

def filler_sentence(language, idx, semantic_seed):
    bank = FILLER_BANK[language]

    x = idx + 7919 * semantic_seed

    subject = bank['subjects'][x % len(bank['subjects'])]
    x //= len(bank['subjects'])

    verb = bank['verbs'][x % len(bank['verbs'])]
    x //= len(bank['verbs'])

    obj = bank['objects'][x % len(bank['objects'])]
    x //= len(bank['objects'])

    setting = bank['settings'][x % len(bank['settings'])]

    if language == 'en':
        return f'{subject} {verb} {obj} {setting}.'

    return f'{subject} {setting} {obj} {verb}.'

def make_records(semantic_seed, trial_index):
    gold = gold_for_trial(trial_index)

    values = DIGITS * (N_RECORDS // 4)

    gold_idx = values.index(gold)

    values[0], values[gold_idx] = (
        values[gold_idx],
        values[0],
    )

    records = [
        (
            make_key(
                semantic_seed,
                slot,
            ),
            values[slot],
        )
        for slot in range(N_RECORDS)
    ]

    target = records[0]
    others = records[1:]

    stable_rng(
        'records_v061',
        semantic_seed,
        trial_index,
    ).shuffle(others)

    pos = target_position_for_trial(trial_index)

    ordered = others[:]
    ordered.insert(pos, target)

    return ordered, target[0], gold, pos

def base_evidence_query(semantic_seed, trial_index):
    records, target_key, gold, pos = make_records(
        semantic_seed,
        trial_index,
    )

    evidence = [
        'Task: look up the value for the queried key.',
        'Each evidence line has the form KEY=VALUE.',
        'Return only the VALUE associated with the queried KEY.',
        'Do not explain or repeat the key.',
        'EVIDENCE:',
    ]

    evidence += [
        f'{key}={value}'
        for key, value in records
    ]

    query = [
        f'QUERY KEY: {target_key}',
        'Answer with one digit only:',
    ]

    return evidence, query, target_key, gold, pos

def compose_user_content(evidence, filler, query):
    evidence_text = '\n'.join(evidence)
    filler_text = '\n'.join(filler)
    query_text = '\n'.join(query)

    user_content = (
        evidence_text
        + '\n'
        + filler_text
        + '\n'
        + query_text
    )

    filler_start = len(evidence_text) + 1
    query_start = filler_start + len(filler_text) + 1

    spans = [
        CharSpan(
            0,
            filler_start,
            'evidence',
        ),
        CharSpan(
            filler_start,
            query_start,
            'filler',
        ),
        CharSpan(
            query_start,
            len(user_content),
            'query',
        ),
    ]

    return user_content, spans

def build_instance(
    filler_language,
    semantic_seed,
    trial_index,
    target_chat_tokens,
):
    (
        evidence,
        query,
        target_key,
        gold,
        pos,
    ) = base_evidence_query(
        semantic_seed,
        trial_index,
    )

    filler = []

    while True:
        user_content, spans = compose_user_content(
            evidence,
            filler,
            query,
        )

        if len(apply_chat_ids(user_content)) >= target_chat_tokens:
            break

        filler.append(
            filler_sentence(
                filler_language,
                len(filler),
                semantic_seed,
            )
        )

        if len(filler) > 5000:
            raise RuntimeError('Filler loop failed.')

    verify_chat_shell(user_content)

    return AssocInstance(
        filler_language=filler_language,
        semantic_seed=semantic_seed,
        trial_index=trial_index,
        target_chat_tokens=target_chat_tokens,
        user_content=user_content,
        spans=spans,
        target_key=target_key,
        gold_digit=gold,
        target_record_position=pos,
        evidence_lines=evidence,
        filler_lines=filler,
        query_lines=query,
    )

print('Generator ready.')

## 7. Exact gold × target-position balance

In [ ]:
balance_rows = []

for trial_index, semantic_seed in enumerate(SEMANTIC_SEEDS):
    balance_rows.append({
        'trial_index': trial_index,
        'semantic_seed': semantic_seed,
        'gold_digit': gold_for_trial(trial_index),
        'position_rank': position_rank_for_trial(trial_index),
        'target_record_position': target_position_for_trial(trial_index),
    })

balance = pd.DataFrame(balance_rows)
display(balance)

gold_position = pd.crosstab(
    balance['gold_digit'],
    balance['position_rank'],
).reindex(
    index=DIGITS,
    columns=POSITION_RANK_SET,
    fill_value=0,
)

display(gold_position)

if not (gold_position == 2).all().all():
    raise RuntimeError(
        'Every gold×position combination must occur exactly twice.'
    )

print('BALANCE GATE PASSED.')

## 8. Canonical and fragmented source-position construction

In [ ]:
def span_label(start, end, spans):
    best_label = None
    best_overlap = 0

    for s in spans:
        overlap = max(
            0,
            min(end, s.end)
            - max(start, s.start),
        )

        if overlap > best_overlap:
            best_overlap = overlap
            best_label = s.label

    if best_label is None:
        raise RuntimeError(
            f'Chunk [{start},{end}) has no semantic region.'
        )

    return best_label

def _piece_intervals(pieces):
    intervals = []
    pos = 0

    for piece in pieces:
        intervals.append(
            (pos, pos + len(piece))
        )
        pos += len(piece)

    return intervals, pos

def _fractional_local_positions(
    canonical_pieces,
    fragmented_pieces,
):
    if ''.join(canonical_pieces) != ''.join(fragmented_pieces):
        raise RuntimeError(
            'Piece streams reconstruct different text.'
        )

    c_int, c_total = _piece_intervals(canonical_pieces)
    f_int, f_total = _piece_intervals(fragmented_pieces)

    if c_total != f_total:
        raise RuntimeError('Character extents disagree.')

    assignment = []
    c_idx = 0

    for fa, fb in f_int:
        while (
            c_idx + 1 < len(c_int)
            and c_int[c_idx][1] <= fa
        ):
            c_idx += 1

        ca, cb = c_int[c_idx]

        if not (ca <= fa and fb <= cb):
            raise RuntimeError(
                'Fragment crossed a canonical-token boundary.'
            )

        assignment.append(c_idx)

    groups = {}

    for fine_idx, coarse_idx in enumerate(assignment):
        groups.setdefault(coarse_idx, []).append(fine_idx)

    positions = [None] * len(fragmented_pieces)

    for coarse_idx, fine_indices in groups.items():
        m = len(fine_indices)

        for rank, fine_idx in enumerate(fine_indices):
            positions[fine_idx] = (
                coarse_idx
                + rank / m
            )

    arr = np.asarray(positions, dtype=np.float32)

    if (
        len(arr) > 1
        and not np.all(np.diff(arr) > 0)
    ):
        raise RuntimeError(
            'Source-aligned local positions are not strictly increasing.'
        )

    return arr.tolist()

def build_user_encoding(inst):
    canonical_ids = []
    fragmented_ids = []

    canonical_labels = []
    fragmented_labels = []

    fragmented_source_positions = []

    canonical_cursor = 0
    dfrag = DIALS[inst.filler_language]

    for chunk, (a, b) in d100.pre_tokenize(inst.user_content):
        label = span_label(a, b, inst.spans)

        c_pieces = d100.encode_chunk_pieces(chunk)
        c_ids = d100.pieces_to_ids(c_pieces)

        canonical_ids.extend(c_ids)
        canonical_labels.extend([label] * len(c_ids))

        if label == 'filler':
            f_pieces = dfrag.encode_chunk_pieces(chunk)
            f_ids = d100.pieces_to_ids(f_pieces)

            local = _fractional_local_positions(
                c_pieces,
                f_pieces,
            )

            src = [
                canonical_cursor + x
                for x in local
            ]

        else:
            f_ids = c_ids
            src = [
                float(x)
                for x in range(
                    canonical_cursor,
                    canonical_cursor + len(c_ids),
                )
            ]

        fragmented_ids.extend(f_ids)
        fragmented_labels.extend([label] * len(f_ids))
        fragmented_source_positions.extend(src)

        canonical_cursor += len(c_ids)

    whole = canonical_user_ids(inst.user_content)

    if canonical_ids != whole:
        raise RuntimeError(
            'Chunkwise canonical encoding mismatch.'
        )

    decoded = tokenizer.decode(
        fragmented_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    if decoded != inst.user_content:
        raise RuntimeError(
            'Fragmented IDs do not decode to identical user content.'
        )

    return {
        'canonical_ids': canonical_ids,
        'fragmented_ids': fragmented_ids,
        'canonical_labels': canonical_labels,
        'fragmented_labels': fragmented_labels,
        'fragmented_source_positions': np.asarray(
            fragmented_source_positions,
            dtype=np.float32,
        ),
    }

print('Encoding helpers ready.')

## 9. The four factorial position maps

`CE` stretches only the **positions** of canonical filler tokens. Its query and assistant suffix positions exactly match `FE`.

Therefore:

- `CC` vs `CE`: same canonical IDs, canonical vs expanded geometry.
- `FC` vs `FE`: same fragmented IDs, source-aligned vs expanded geometry.

In [ ]:
def hash_ids(ids):
    return hashlib.sha256(
        np.asarray(
            ids,
            dtype=np.int64,
        ).tobytes()
    ).hexdigest()[:24]

def hash_positions(pos):
    return hashlib.sha256(
        np.asarray(
            pos,
            dtype=np.float32,
        ).tobytes()
    ).hexdigest()[:24]

def full_gap_diagnostics(
    positions,
    prefix_len,
    labels,
):
    labels = np.asarray(labels)

    evidence_idx = np.where(
        labels == 'evidence'
    )[0]

    filler_idx = np.where(
        labels == 'filler'
    )[0]

    query_idx = np.where(
        labels == 'query'
    )[0]

    if not (
        len(evidence_idx)
        and len(filler_idx)
        and len(query_idx)
    ):
        raise RuntimeError(
            'Missing evidence/filler/query region.'
        )

    evidence_last = float(
        positions[
            prefix_len + evidence_idx[-1]
        ]
    )

    filler_first = float(
        positions[
            prefix_len + filler_idx[0]
        ]
    )

    filler_last = float(
        positions[
            prefix_len + filler_idx[-1]
        ]
    )

    query_first = float(
        positions[
            prefix_len + query_idx[0]
        ]
    )

    query_last = float(
        positions[
            prefix_len + query_idx[-1]
        ]
    )

    readout = float(
        positions[-1]
    )

    return {
        'evidence_query': (
            query_first - evidence_last
        ),
        'evidence_readout': (
            readout - evidence_last
        ),
        'filler_span': (
            filler_last - filler_first
        ),
        'query_readout': (
            readout - query_last
        ),
    }

def assemble_factorial(inst):
    shell = verify_chat_shell(
        inst.user_content
    )

    enc = build_user_encoding(inst)

    c_user = enc['canonical_ids']
    f_user = enc['fragmented_ids']

    c_labels = np.asarray(
        enc['canonical_labels']
    )

    f_labels = np.asarray(
        enc['fragmented_labels']
    )

    prefix_len = len(CHAT_PREFIX_IDS)
    suffix_len = len(CHAT_SUFFIX_IDS)

    c_full_ids = (
        CHAT_PREFIX_IDS
        + c_user
        + CHAT_SUFFIX_IDS
    )

    f_full_ids = (
        CHAT_PREFIX_IDS
        + f_user
        + CHAT_SUFFIX_IDS
    )

    if c_full_ids != shell['official_ids']:
        raise RuntimeError(
            'Canonical full IDs disagree with official chat.'
        )

    # CC: canonical IDs + ordinary canonical positions.
    cc_pos = np.arange(
        len(c_full_ids),
        dtype=np.float32,
    )

    # FE: fragmented IDs + ordinary expanded positions.
    fe_pos = np.arange(
        len(f_full_ids),
        dtype=np.float32,
    )

    # FC: fragmented IDs + source-aligned canonical positions.
    prefix_pos = np.arange(
        prefix_len,
        dtype=np.float32,
    )

    fc_user_pos = (
        prefix_len
        + enc[
            'fragmented_source_positions'
        ]
    )

    canonical_suffix_start = (
        prefix_len
        + len(c_user)
    )

    fc_suffix_pos = np.arange(
        canonical_suffix_start,
        canonical_suffix_start + suffix_len,
        dtype=np.float32,
    )

    fc_pos = np.concatenate(
        [
            prefix_pos,
            fc_user_pos,
            fc_suffix_pos,
        ]
    ).astype(np.float32)

    # CE: canonical IDs, but stretch the canonical filler positional span
    # so that the first query token and assistant suffix match FE.
    ce_user_pos = np.arange(
        len(c_user),
        dtype=np.float32,
    )

    c_filler_idx = np.where(
        c_labels == 'filler'
    )[0]

    c_query_idx = np.where(
        c_labels == 'query'
    )[0]

    f_query_idx = np.where(
        f_labels == 'query'
    )[0]

    if not (
        len(c_filler_idx)
        and len(c_query_idx)
        and len(f_query_idx)
    ):
        raise RuntimeError('Missing canonical/fragmented regions.')

    c_filler_first = int(
        c_filler_idx[0]
    )

    c_filler_n = len(
        c_filler_idx
    )

    f_query_first = int(
        f_query_idx[0]
    )

    # Fill [canonical filler start, fragmented query start) uniformly.
    ce_user_pos[
        c_filler_idx
    ] = (
        c_filler_first
        + np.arange(
            c_filler_n,
            dtype=np.float32,
        )
        * (
            f_query_first
            - c_filler_first
        )
        / c_filler_n
    )

    # Query tokens are placed exactly where the corresponding FE query
    # tokens begin/end; query tokenization itself is unchanged.
    c_query_first = int(
        c_query_idx[0]
    )

    query_offset = (
        f_query_first
        - c_query_first
    )

    ce_user_pos[
        c_query_idx
    ] = (
        c_query_idx.astype(
            np.float32
        )
        + query_offset
    )

    expanded_suffix_start = (
        prefix_len
        + len(f_user)
    )

    ce_suffix_pos = np.arange(
        expanded_suffix_start,
        expanded_suffix_start + suffix_len,
        dtype=np.float32,
    )

    ce_pos = np.concatenate(
        [
            prefix_pos,
            prefix_len + ce_user_pos,
            ce_suffix_pos,
        ]
    ).astype(np.float32)

    maps = {
        'CC': {
            'condition': 'canonical_ids_canonical_pos',
            'ids': c_full_ids,
            'positions': cc_pos,
            'labels': c_labels,
        },
        'CE': {
            'condition': 'canonical_ids_expanded_pos',
            'ids': c_full_ids,
            'positions': ce_pos,
            'labels': c_labels,
        },
        'FC': {
            'condition': 'fragmented_ids_source_pos',
            'ids': f_full_ids,
            'positions': fc_pos,
            'labels': f_labels,
        },
        'FE': {
            'condition': 'fragmented_ids_expanded_pos',
            'ids': f_full_ids,
            'positions': fe_pos,
            'labels': f_labels,
        },
    }

    for key, item in maps.items():
        p = item['positions']

        if len(item['ids']) != len(p):
            raise RuntimeError(
                f'{key}: IDs/positions length mismatch.'
            )

        if (
            len(p) > 1
            and not np.all(np.diff(p) > 0)
        ):
            bad = np.where(
                np.diff(p) <= 0
            )[0][:10]

            raise RuntimeError(
                f'{key}: positions not strictly increasing: {bad.tolist()}'
            )

        item['ids_hash'] = hash_ids(
            item['ids']
        )

        item['position_hash'] = hash_positions(
            p
        )

        item['gaps'] = full_gap_diagnostics(
            p,
            prefix_len,
            item['labels'],
        )

        item['realized_tokens'] = len(
            item['ids']
        )

        item['position_max'] = float(
            p[-1]
        )

    # Exact factorial invariants.
    if maps['CC']['ids_hash'] != maps['CE']['ids_hash']:
        raise RuntimeError(
            'CC and CE must have identical canonical token IDs.'
        )

    if maps['FC']['ids_hash'] != maps['FE']['ids_hash']:
        raise RuntimeError(
            'FC and FE must have identical fragmented token IDs.'
        )

    if abs(
        maps['CC']['gaps']['evidence_readout']
        - maps['FC']['gaps']['evidence_readout']
    ) > 1e-3:
        raise RuntimeError(
            'CC and FC must share canonical global evidence→readout geometry.'
        )

    if abs(
        maps['CE']['gaps']['evidence_readout']
        - maps['FE']['gaps']['evidence_readout']
    ) > 1e-3:
        raise RuntimeError(
            'CE and FE must share expanded global evidence→readout geometry.'
        )

    if not (
        maps['FE']['gaps']['evidence_readout']
        > maps['FC']['gaps']['evidence_readout']
    ):
        raise RuntimeError(
            'Expanded geometry did not increase evidence→readout distance.'
        )

    return maps

print('Factorial assembly ready.')

## 10. T4-safe scorer with raw candidate logit margins

In [ ]:
def _sequential_chunk_mask(start, end, device):
    q = torch.arange(
        start,
        end,
        device=device,
    )[:, None]

    k = torch.arange(
        0,
        end,
        device=device,
    )[None, :]

    return (
        k <= q
    )[None, None, :, :]

def finish_scores(h_last, gold_digit):
    logits = model.lm_head(
        h_last
    ).float()[0]

    candidate_ids = torch.tensor(
        [
            DIGIT_IDS[d]
            for d in DIGITS
        ],
        dtype=torch.long,
        device=logits.device,
    )

    candidate_logits = logits.index_select(
        0,
        candidate_ids,
    )

    candidate_logp = F.log_softmax(
        candidate_logits,
        dim=-1,
    )

    candidate_prob = candidate_logp.exp()

    gold_idx = DIGITS.index(
        gold_digit
    )

    distractor_indices = [
        i
        for i in range(len(DIGITS))
        if i != gold_idx
    ]

    best_distractor_idx = max(
        distractor_indices,
        key=lambda i: float(
            candidate_logits[i].item()
        ),
    )

    gold_margin = float(
        (
            candidate_logits[gold_idx]
            - candidate_logits[best_distractor_idx]
        ).item()
    )

    pred_idx = int(
        torch.argmax(
            candidate_logits
        ).item()
    )

    pred = DIGITS[pred_idx]

    full_argmax_id = int(
        torch.argmax(logits).item()
    )

    full_argmax_digit = next(
        (
            d
            for d, tok_id in DIGIT_IDS.items()
            if tok_id == full_argmax_id
        ),
        None,
    )

    result = {
        'predicted_digit': pred,
        'correct': int(
            pred == gold_digit
        ),
        'gold_logit_margin': gold_margin,
        'gold_logit': float(
            candidate_logits[gold_idx].item()
        ),
        'best_distractor_digit': DIGITS[
            best_distractor_idx
        ],
        'best_distractor_logit': float(
            candidate_logits[
                best_distractor_idx
            ].item()
        ),
        'forced_choice_nll': float(
            -candidate_logp[
                gold_idx
            ].item()
        ),
        'gold_candidate_probability': float(
            candidate_prob[
                gold_idx
            ].item()
        ),
        'full_argmax_id': full_argmax_id,
        'full_argmax_text': tokenizer.decode(
            [full_argmax_id],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ),
        'full_argmax_digit': full_argmax_digit,
        'full_argmax_is_gold': int(
            full_argmax_digit == gold_digit
        ),
        'candidate_logits': {
            d: float(
                candidate_logits[i].item()
            )
            for i, d in enumerate(DIGITS)
        },
        'candidate_probabilities': {
            d: float(
                candidate_prob[i].item()
            )
            for i, d in enumerate(DIGITS)
        },
    }

    del (
        logits,
        candidate_ids,
        candidate_logits,
        candidate_logp,
        candidate_prob,
    )

    return result

def score_chunked(
    input_ids,
    position_ids,
    gold_digit,
    chunk_size=PREFILL_CHUNK,
):
    if len(input_ids) != len(position_ids):
        raise ValueError('IDs/positions length mismatch.')

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    ids_all = torch.tensor(
        input_ids,
        dtype=torch.long,
        device=device,
    )

    pos_all = torch.tensor(
        position_ids,
        dtype=torch.float32,
        device=device,
    )

    cache = DynamicCache(
        config=model.config
    )

    last_hidden = None

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.inference_mode():
        n = ids_all.numel()

        for start in range(
            0,
            n,
            chunk_size,
        ):
            end = min(
                start + chunk_size,
                n,
            )

            ids_chunk = ids_all[
                start:end
            ].unsqueeze(0)

            pos_chunk = pos_all[
                start:end
            ].unsqueeze(0)

            causal = _sequential_chunk_mask(
                start,
                end,
                device,
            )

            out = backbone(
                input_ids=ids_chunk,
                attention_mask={
                    'full_attention': causal
                },
                position_ids=pos_chunk,
                past_key_values=cache,
                use_cache=True,
                return_dict=True,
            )

            cache = out.past_key_values

            if end == n:
                last_hidden = out.last_hidden_state[
                    :, -1, :
                ]

            del (
                ids_chunk,
                pos_chunk,
                causal,
                out,
            )

        scored = finish_scores(
            last_hidden,
            gold_digit,
        )

    torch.cuda.synchronize()

    scored['seconds'] = (
        time.perf_counter() - t0
    )

    scored['peak_cuda_gb'] = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    scored['scorer_version'] = SCORER_VERSION

    del (
        ids_all,
        pos_all,
        cache,
        last_hidden,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return scored

def flatten_score(scored):
    out = {
        k: v
        for k, v in scored.items()
        if k not in {
            'candidate_logits',
            'candidate_probabilities',
        }
    }

    for d, v in scored['candidate_logits'].items():
        out[f'logit_{d}'] = v

    for d, v in scored['candidate_probabilities'].items():
        out[f'prob_{d}'] = v

    return out

print('Margin scorer ready.')

## 11. Structural engineering gate at ~1K

In [ ]:
gate_inst = build_instance(
    filler_language='en',
    semantic_seed=2999,
    trial_index=7,
    target_chat_tokens=1024,
)

maps = assemble_factorial(gate_inst)

for key in ['CC', 'CE', 'FC', 'FE']:
    item = maps[key]

    print(
        key,
        {
            'condition': item['condition'],
            'tokens': item['realized_tokens'],
            'position_max': item['position_max'],
            'evidence_readout_gap': item['gaps']['evidence_readout'],
            'ids_hash': item['ids_hash'],
            'position_hash': item['position_hash'],
        },
    )

print(
    'Geometry pairs:',
    {
        'source_gap': (
            maps['CC']['gaps']['evidence_readout'],
            maps['FC']['gaps']['evidence_readout'],
        ),
        'expanded_gap': (
            maps['CE']['gaps']['evidence_readout'],
            maps['FE']['gaps']['evidence_readout'],
        ),
    },
)

gate_scores = {}

for key in ['CC', 'CE', 'FC', 'FE']:
    gate_scores[key] = score_chunked(
        maps[key]['ids'],
        maps[key]['positions'],
        gate_inst.gold_digit,
    )

    print(
        key,
        {
            'pred': gate_scores[key]['predicted_digit'],
            'margin': gate_scores[key]['gold_logit_margin'],
            'nll': gate_scores[key]['forced_choice_nll'],
            'full': gate_scores[key]['full_argmax_text'],
            'peak_gb': gate_scores[key]['peak_cuda_gb'],
        },
    )

    if not (
        np.isfinite(
            gate_scores[key]['gold_logit_margin']
        )
        and np.isfinite(
            gate_scores[key]['forced_choice_nll']
        )
    ):
        raise RuntimeError(
            f'Non-finite engineering score in {key}.'
        )

print('STRUCTURAL ENGINEERING GATE PASSED.')

## 12. Resumable JSONL utilities

In [ ]:
def load_jsonl(path):
    rows = []

    if path.exists():
        with path.open(
            'r',
            encoding='utf-8',
        ) as f:
            for line in f:
                line = line.strip()

                if line:
                    rows.append(
                        json.loads(line)
                    )

    return rows

def completed_ids(path):
    return {
        row['result_id']
        for row in load_jsonl(path)
        if 'result_id' in row
    }

def make_result_id(meta):
    blob = json.dumps(
        meta,
        sort_keys=True,
        separators=(',', ':'),
    ).encode('utf-8')

    return hashlib.sha256(
        blob
    ).hexdigest()[:24]

def append_jsonl(path, row):
    with path.open(
        'a',
        encoding='utf-8',
    ) as f:
        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + '\n'
        )
        f.flush()

# Phase A — prospective 2×2 factorial run

32 trials × 2 filler languages × 2 lengths × 4 conditions = **512 rows**.

Condition order is deterministically randomized inside every base instance.

In [ ]:
CONDITIONS = ['CC', 'CE', 'FC', 'FE']

done = completed_ids(
    RESULTS_PATH
)

for filler_language in FILLER_LANGUAGES:
    for target_chat_tokens in TARGET_LENGTHS:
        for trial_index, semantic_seed in enumerate(
            SEMANTIC_SEEDS
        ):
            inst = build_instance(
                filler_language=filler_language,
                semantic_seed=semantic_seed,
                trial_index=trial_index,
                target_chat_tokens=target_chat_tokens,
            )

            maps = assemble_factorial(inst)

            order = CONDITIONS[:]

            stable_rng(
                'condition_order_v061',
                filler_language,
                target_chat_tokens,
                semantic_seed,
                trial_index,
            ).shuffle(order)

            evidence_text = '\n'.join(
                inst.evidence_lines
            )

            query_text = '\n'.join(
                inst.query_lines
            )

            for order_index, key in enumerate(order):
                item = maps[key]

                meta = {
                    'phase': 'factorial_4k8k',
                    'model': MODEL_NAME,
                    'task_version': TASK_VERSION,
                    'scorer_version': SCORER_VERSION,
                    'position_intervention': POSITION_INTERVENTION,
                    'filler_language': filler_language,
                    'target_chat_tokens': target_chat_tokens,
                    'semantic_seed': semantic_seed,
                    'trial_index': trial_index,
                    'n_records': N_RECORDS,
                    'factorial_code': key,
                    'condition': item['condition'],
                    'frag_keep': (
                        None
                        if key in {'CC', 'CE'}
                        else FRAG_KEEP_BY_LANGUAGE[
                            filler_language
                        ]
                    ),
                }

                rid = make_result_id(meta)

                if rid in done:
                    continue

                scored = score_chunked(
                    item['ids'],
                    item['positions'],
                    inst.gold_digit,
                )

                row = {
                    'result_id': rid,
                    **meta,
                    'condition_order_index': order_index,
                    'gold_digit': inst.gold_digit,
                    'target_key': inst.target_key,
                    'position_rank': position_rank_for_trial(
                        trial_index
                    ),
                    'target_record_position': inst.target_record_position,
                    'realized_full_tokens': item[
                        'realized_tokens'
                    ],
                    'position_max': item[
                        'position_max'
                    ],
                    'evidence_query_position_gap': item[
                        'gaps'
                    ][
                        'evidence_query'
                    ],
                    'evidence_readout_position_gap': item[
                        'gaps'
                    ][
                        'evidence_readout'
                    ],
                    'filler_position_span': item[
                        'gaps'
                    ][
                        'filler_span'
                    ],
                    'ids_hash': item[
                        'ids_hash'
                    ],
                    'position_hash': item[
                        'position_hash'
                    ],
                    'prompt_sha256': hashlib.sha256(
                        inst.user_content.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'evidence_sha256': hashlib.sha256(
                        evidence_text.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'query_sha256': hashlib.sha256(
                        query_text.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
                    **flatten_score(scored),
                }

                append_jsonl(
                    RESULTS_PATH,
                    row,
                )

                done.add(rid)

                print(
                    f"fill={filler_language} "
                    f"L={target_chat_tokens:5d} "
                    f"trial={trial_index:02d} "
                    f"{key} "
                    f"gold={inst.gold_digit} "
                    f"pred={scored['predicted_digit']} "
                    f"margin={scored['gold_logit_margin']:+8.3f} "
                    f"NLL={scored['forced_choice_nll']:.5f} "
                    f"n={item['realized_tokens']:5d} "
                    f"gap={item['gaps']['evidence_readout']:8.1f} "
                    f"peak={scored['peak_cuda_gb']:.2f}GB"
                )

print('Factorial run complete:', RESULTS_PATH)

## 13. Pairing and factorial-invariant gates

In [ ]:
df = pd.DataFrame(
    load_jsonl(
        RESULTS_PATH
    )
)

base_cols = [
    'filler_language',
    'target_chat_tokens',
    'semantic_seed',
    'trial_index',
]

condition_counts = df.groupby(
    base_cols
)['factorial_code'].nunique()

if (condition_counts != 4).any():
    raise RuntimeError(
        'Some base instances do not have all four factorial conditions.'
    )

if (
    df.groupby(
        base_cols
    )[
        'prompt_sha256'
    ].nunique()
    != 1
).any():
    raise RuntimeError(
        'Prompt text differs across factorial conditions.'
    )

pair = df.pivot_table(
    index=base_cols,
    columns='factorial_code',
    values=[
        'ids_hash',
        'evidence_readout_position_gap',
        'position_max',
    ],
    aggfunc='first',
)

if not (
    pair['ids_hash']['CC']
    == pair['ids_hash']['CE']
).all():
    raise RuntimeError(
        'CC/CE token IDs differ.'
    )

if not (
    pair['ids_hash']['FC']
    == pair['ids_hash']['FE']
).all():
    raise RuntimeError(
        'FC/FE token IDs differ.'
    )

if (
    np.abs(
        pair[
            'evidence_readout_position_gap'
        ]['CC']
        - pair[
            'evidence_readout_position_gap'
        ]['FC']
    ).max()
    > 1e-3
):
    raise RuntimeError(
        'CC/FC source geometry differs.'
    )

if (
    np.abs(
        pair[
            'evidence_readout_position_gap'
        ]['CE']
        - pair[
            'evidence_readout_position_gap'
        ]['FE']
    ).max()
    > 1e-3
):
    raise RuntimeError(
        'CE/FE expanded geometry differs.'
    )

if not (
    pair[
        'evidence_readout_position_gap'
    ]['FE']
    > pair[
        'evidence_readout_position_gap'
    ]['FC']
).all():
    raise RuntimeError(
        'Expanded geometry is not larger in every pair.'
    )

task_pair_cols = [
    'target_chat_tokens',
    'semantic_seed',
    'trial_index',
]

if (
    df.groupby(
        task_pair_cols
    )['evidence_sha256'].nunique()
    != 1
).any():
    raise RuntimeError(
        'Task evidence differs across filler languages.'
    )

if (
    df.groupby(
        task_pair_cols
    )['query_sha256'].nunique()
    != 1
).any():
    raise RuntimeError(
        'Task query differs across filler languages.'
    )

print('PAIRING / SAME-ID / MATCHED-GEOMETRY GATES PASSED.')

## 14. Performance summary

In [ ]:
performance = df.groupby(
    [
        'filler_language',
        'target_chat_tokens',
        'factorial_code',
        'condition',
    ],
    as_index=False,
).agg(
    n=('correct', 'size'),
    accuracy=('correct', 'mean'),
    mean_margin=('gold_logit_margin', 'mean'),
    median_margin=('gold_logit_margin', 'median'),
    mean_forced_nll=('forced_choice_nll', 'mean'),
    mean_gold_probability=(
        'gold_candidate_probability',
        'mean',
    ),
    full_argmax_gold_rate=(
        'full_argmax_is_gold',
        'mean',
    ),
    mean_tokens=('realized_full_tokens', 'mean'),
    mean_position_max=('position_max', 'mean'),
    mean_evidence_readout_gap=(
        'evidence_readout_position_gap',
        'mean',
    ),
    max_peak_cuda_gb=('peak_cuda_gb', 'max'),
)

display(performance)

# Factorial effects

## 15. Paired logit-margin contrasts

All effects are defined so **positive values mean better gold-vs-distractor separation**.

- `position_effect_canonical_ids = CE - CC`
- `position_effect_fragmented_ids = FE - FC`
- `representation_effect_source_geometry = FC - CC`
- `representation_effect_expanded_geometry = FE - CE`
- `position_interaction = (FE - FC) - (CE - CC)`
- `total_fragmentation_shift = FE - CC`

In [ ]:
wide = df.pivot_table(
    index=base_cols,
    columns='factorial_code',
    values=[
        'gold_logit_margin',
        'forced_choice_nll',
        'correct',
    ],
    aggfunc='first',
).reset_index()

for code_name in ['CC', 'CE', 'FC', 'FE']:
    if (
        'gold_logit_margin',
        code_name,
    ) not in wide.columns:
        raise RuntimeError(
            f'Missing {code_name} margin column.'
        )

effects = pd.DataFrame({
    'filler_language': wide[
        ('filler_language', '')
    ],
    'target_chat_tokens': wide[
        ('target_chat_tokens', '')
    ],
    'semantic_seed': wide[
        ('semantic_seed', '')
    ],
    'trial_index': wide[
        ('trial_index', '')
    ],
})

CC = wide[
    ('gold_logit_margin', 'CC')
].to_numpy(float)

CE = wide[
    ('gold_logit_margin', 'CE')
].to_numpy(float)

FC = wide[
    ('gold_logit_margin', 'FC')
].to_numpy(float)

FE = wide[
    ('gold_logit_margin', 'FE')
].to_numpy(float)

effects[
    'position_effect_canonical_ids'
] = CE - CC

effects[
    'position_effect_fragmented_ids'
] = FE - FC

effects[
    'representation_effect_source_geometry'
] = FC - CC

effects[
    'representation_effect_expanded_geometry'
] = FE - CE

effects[
    'position_interaction'
] = (
    (FE - FC)
    - (CE - CC)
)

effects[
    'total_fragmentation_shift'
] = FE - CC

effects[
    'CC_margin'
] = CC

effects[
    'CE_margin'
] = CE

effects[
    'FC_margin'
] = FC

effects[
    'FE_margin'
] = FE

display(effects.head(20))

## 16. Cluster bootstrap and cell-wise replication

In [ ]:
EFFECT_NAMES = [
    'position_effect_canonical_ids',
    'position_effect_fragmented_ids',
    'representation_effect_source_geometry',
    'representation_effect_expanded_geometry',
    'position_interaction',
    'total_fragmentation_shift',
]

def bootstrap_mean_ci(
    values,
    n_boot=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
):
    x = np.asarray(
        pd.Series(values).dropna(),
        dtype=float,
    )

    rng = np.random.default_rng(seed)

    idx = rng.integers(
        0,
        len(x),
        size=(n_boot, len(x)),
    )

    boot = x[idx].mean(axis=1)

    return (
        float(x.mean()),
        float(np.quantile(boot, 0.025)),
        float(np.quantile(boot, 0.975)),
    )

def cluster_bootstrap_mean_ci(
    frame,
    value_col,
    cluster_col='trial_index',
    n_boot=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
):
    clean = frame[
        [cluster_col, value_col]
    ].dropna()

    clusters = np.asarray(
        sorted(
            clean[cluster_col].unique()
        )
    )

    grouped = {
        c: clean.loc[
            clean[cluster_col] == c,
            value_col,
        ].to_numpy(dtype=float)
        for c in clusters
    }

    observed = float(
        clean[value_col].mean()
    )

    rng = np.random.default_rng(seed)

    boot = np.empty(n_boot)

    for b in range(n_boot):
        sampled = rng.choice(
            clusters,
            size=len(clusters),
            replace=True,
        )

        vals = np.concatenate(
            [grouped[c] for c in sampled]
        )

        boot[b] = vals.mean()

    return (
        observed,
        float(np.quantile(boot, 0.025)),
        float(np.quantile(boot, 0.975)),
    )

cell_rows = []

for (
    filler_language,
    target_chat_tokens
), g in effects.groupby(
    [
        'filler_language',
        'target_chat_tokens',
    ]
):
    for effect in EFFECT_NAMES:
        mean, lo, hi = bootstrap_mean_ci(
            g[effect],
            seed=(
                BOOTSTRAP_SEED
                + stable_int(
                    filler_language,
                    target_chat_tokens,
                    effect,
                )
                % 100000
            ),
        )

        cell_rows.append({
            'filler_language': filler_language,
            'target_chat_tokens': int(
                target_chat_tokens
            ),
            'effect': effect,
            'n': len(g),
            'mean': mean,
            'ci_lo': lo,
            'ci_hi': hi,
            'positive_n': int(
                (g[effect] > 0).sum()
            ),
            'negative_n': int(
                (g[effect] < 0).sum()
            ),
        })

cell_summary = pd.DataFrame(
    cell_rows
)

display(cell_summary)

pooled_rows = []

for effect in EFFECT_NAMES:
    mean, lo, hi = cluster_bootstrap_mean_ci(
        effects,
        effect,
        seed=(
            BOOTSTRAP_SEED
            + stable_int(
                'pooled',
                effect,
            )
            % 100000
        ),
    )

    pooled_rows.append({
        'effect': effect,
        'n_rows': len(effects),
        'n_trial_clusters': effects[
            'trial_index'
        ].nunique(),
        'mean': mean,
        'ci_lo': lo,
        'ci_hi': hi,
        'positive_n': int(
            (effects[effect] > 0).sum()
        ),
        'negative_n': int(
            (effects[effect] < 0).sum()
        ),
    })

pooled_summary = pd.DataFrame(
    pooled_rows
)

display(pooled_summary)

for _, row in pooled_summary.iterrows():
    print(
        f"{row['effect']:<42} "
        f"mean={row['mean']:+.4f} "
        f"95% cluster-CI=[{row['ci_lo']:+.4f},{row['ci_hi']:+.4f}] "
        f"positive={int(row['positive_n'])}/{int(row['n_rows'])}"
    )

## 17. Leave-one-trial-out influence audit

This is predeclared because v0.6.0's forced-choice-NLL effects were highly influenced by a small number of informative items.

A follow-up effect should not depend on one semantic trial.

In [ ]:
loo_rows = []

for left_out in sorted(
    effects['trial_index'].unique()
):
    g = effects[
        effects['trial_index']
        != left_out
    ]

    row = {
        'left_out_trial': int(left_out),
    }

    for effect in EFFECT_NAMES:
        row[effect] = float(
            g[effect].mean()
        )

    loo_rows.append(row)

loo = pd.DataFrame(
    loo_rows
)

display(loo)

loo_summary = []

for effect in EFFECT_NAMES:
    loo_summary.append({
        'effect': effect,
        'full_mean': float(
            effects[effect].mean()
        ),
        'loo_min': float(
            loo[effect].min()
        ),
        'loo_max': float(
            loo[effect].max()
        ),
        'loo_sign_stable_positive': bool(
            (
                loo[effect] > 0
            ).all()
        ),
        'loo_sign_stable_negative': bool(
            (
                loo[effect] < 0
            ).all()
        ),
    })

loo_summary = pd.DataFrame(
    loo_summary
)

display(loo_summary)

## 18. Position-rank and language×length diagnostics

These are descriptive generality checks. They are not used to select a subset of trials.

In [ ]:
meta = balance[
    [
        'trial_index',
        'position_rank',
        'target_record_position',
        'gold_digit',
    ]
]

effects_meta = effects.merge(
    meta,
    on='trial_index',
    how='left',
)

rank_summary = effects_meta.groupby(
    'position_rank',
    as_index=False,
)[
    [
        'position_effect_canonical_ids',
        'position_effect_fragmented_ids',
        'representation_effect_source_geometry',
        'position_interaction',
    ]
].mean()

display(rank_summary)

cell_wide = cell_summary.pivot_table(
    index=[
        'filler_language',
        'target_chat_tokens',
    ],
    columns='effect',
    values='mean',
    aggfunc='first',
).reset_index()

display(cell_wide)

# Prospective diagnostic decision

## 19. Factorial replication gate

The v0.6.0 observation motivating this follow-up was that expanded normal positions often improved separation rather than degrading it.

A **position-expansion replication** requires:

- canonical (`CC`) forced-choice accuracy ≥ 90%;
- no collapsed answer prior;
- pooled `CE - CC > 0`;
- pooled `FE - FC > 0`;
- each position effect is positive in at least 3/4 language×length cells;
- each position effect remains positive under every leave-one-trial-out estimate.

`FACTORIAL_POSITION_REPLICATION=True` would confirm the revised finding on fresh data.

It does **not** resurrect the original degradation/rescue hypothesis.

In [ ]:
cc_rows = df[
    df['factorial_code']
    == 'CC'
]

canonical_acc = float(
    cc_rows['correct'].mean()
)

cc_pred_counts = (
    cc_rows['predicted_digit']
    .value_counts()
    .reindex(
        DIGITS,
        fill_value=0,
    )
)

canonical_max_pred_share = float(
    cc_pred_counts.max()
    / cc_pred_counts.sum()
)

pooled = pooled_summary.set_index(
    'effect'
)

pos_can = float(
    pooled.loc[
        'position_effect_canonical_ids',
        'mean',
    ]
)

pos_frag = float(
    pooled.loc[
        'position_effect_fragmented_ids',
        'mean',
    ]
)

positive_cells_can = int(
    (
        cell_wide[
            'position_effect_canonical_ids'
        ] > 0
    ).sum()
)

positive_cells_frag = int(
    (
        cell_wide[
            'position_effect_fragmented_ids'
        ] > 0
    ).sum()
)

loo_can = bool(
    (
        loo[
            'position_effect_canonical_ids'
        ] > 0
    ).all()
)

loo_frag = bool(
    (
        loo[
            'position_effect_fragmented_ids'
        ] > 0
    ).all()
)

FACTORIAL_POSITION_REPLICATION = bool(
    canonical_acc >= 0.90
    and canonical_max_pred_share <= 0.40
    and pos_can > 0
    and pos_frag > 0
    and positive_cells_can >= 3
    and positive_cells_frag >= 3
    and loo_can
    and loo_frag
)

print('CC accuracy:', canonical_acc)
print(
    'CC prediction counts:',
    cc_pred_counts.to_dict(),
)
print(
    'CC max prediction share:',
    canonical_max_pred_share,
)
print(
    'Pooled CE-CC position effect:',
    pos_can,
)
print(
    'Pooled FE-FC position effect:',
    pos_frag,
)
print(
    'Positive language×length cells:',
    {
        'canonical_ids': positive_cells_can,
        'fragmented_ids': positive_cells_frag,
    },
)
print(
    'Leave-one-trial-out sign stability:',
    {
        'canonical_ids': loo_can,
        'fragmented_ids': loo_frag,
    },
)

print(
    '\nFACTORIAL_POSITION_REPLICATION =',
    FACTORIAL_POSITION_REPLICATION,
)

if FACTORIAL_POSITION_REPLICATION:
    print(
        '\nCONFIRMED ON NEW SEEDS: expanded positional geometry improves '
        'gold-vs-distractor separation in this associative-retrieval substrate. '
        'The paper should be reframed around positional distortion / behavioral '
        'change rather than a universal degradation claim.'
    )
else:
    print(
        '\nNOT REPLICATED: do not reinterpret v0.6.0 as a stable positional '
        'benefit. The sign-reversal observation remains exploratory.'
    )

## 20. Save audit

In [ ]:
pooled_effects = {}

for _, row in pooled_summary.iterrows():
    pooled_effects[
        row['effect']
    ] = {
        'mean': float(
            row['mean']
        ),
        'ci_lo': float(
            row['ci_lo']
        ),
        'ci_hi': float(
            row['ci_hi']
        ),
    }

audit = {
    'model': MODEL_NAME,
    'transformers_version': transformers.__version__,
    'notebook_version': 'v0.6.1',
    'task_version': TASK_VERSION,
    'scorer_version': SCORER_VERSION,
    'position_intervention': POSITION_INTERVENTION,
    'followup_status': 'prospective_new_seed_diagnostic',
    'original_v060_mechanism_go': (
        None
        if previous is None
        else previous.get('mechanism_go')
    ),
    'claim_scope': 'associative_retrieval',
    'frozen_n_records': N_RECORDS,
    'frozen_frag_keep_by_language': FRAG_KEEP_BY_LANGUAGE,
    'new_semantic_seeds': SEMANTIC_SEEDS,
    'target_lengths': TARGET_LENGTHS,
    'primary_metric': 'gold_vs_best_distractor_logit_margin',
    'gold_position_combinations_each_n': 2,
    'same_ids_CC_CE': True,
    'same_ids_FC_FE': True,
    'matched_source_geometry_CC_FC': True,
    'matched_expanded_geometry_CE_FE': True,
    'canonical_accuracy': canonical_acc,
    'canonical_max_pred_share': canonical_max_pred_share,
    'pooled_effects_cluster_bootstrap': pooled_effects,
    'positive_cells': {
        'position_effect_canonical_ids': positive_cells_can,
        'position_effect_fragmented_ids': positive_cells_frag,
    },
    'leave_one_trial_out_sign_stable': {
        'position_effect_canonical_ids': loo_can,
        'position_effect_fragmented_ids': loo_frag,
    },
    'factorial_position_replication': bool(
        FACTORIAL_POSITION_REPLICATION
    ),
}

with AUDIT_PATH.open(
    'w',
    encoding='utf-8',
) as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    )
)

print('Saved:', AUDIT_PATH)

# Files to upload

From:

`MyDrive/qwen3_position_factorial_v061/`

upload:

1. `v061_factorial_4k8k.jsonl`
2. `v061_audit.json`

Key line:

`FACTORIAL_POSITION_REPLICATION = True/False`

Do not proceed to evidence-locus fragmentation or another model until this follow-up is reviewed.